In [2]:
from llama_index.core import (
    VectorStoreIndex,
    SummaryIndex,
    SimpleDirectoryReader,
    ServiceContext,
)
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI

In [3]:
wiki_titles = [
    "Serie A",
    "Premier League",
    "Bundesliga",
    "La Liga",
    "Ligue 1"
]

In [4]:
from pathlib import Path

import requests

for title in wiki_titles:
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "titles": title,
            "prop": "extracts",
            "explaintext": True,
        },
    ).json()
    page = next(iter(response["query"]["pages"].values()))
    wiki_text = page["extract"]

    data_path = Path("data")
    if not data_path.exists():
        Path.mkdir(data_path)

    with open(data_path / f"{title}.txt", "w") as fp:
        fp.write(wiki_text)

In [5]:
leagues_docs = {}
for wiki_title in wiki_titles:
    leagues_docs[wiki_title] = SimpleDirectoryReader(
        input_files=[f"data/{wiki_title}.txt"]
    ).load_data()

In [10]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

In [11]:
Settings.llm = OpenAI(model="gpt-3.5-turbo")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 3900

In [23]:
class Element(BaseModel):
    type: str
    text: Any

table_elements = []
text_elements = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        table_elements.append(Element(type="table", text=str(element)))
    elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
        text_elements.append(Element(type="text", text=str(element)))

In [26]:
table_elements[0]

Element(type='table', text='NASDAQ 18196 -3.43 -5.66 -5.66 12.60 45.25 25.66 6.52 0.68 29729 Levels Fixed Income Yield 1 week QTD YTD 1year 3-yr.Cum. Currencies 3/7/25 12/31/24 3/7/24 U.S. Aggregate 4.67 -0.58 2.15 2.15 4.08 1.60 $per€ 1.09 1.04 1.09 U.S. Corporates 5.18 -0.65 1.93 1.93 4.66 149° $perf 1.29 1.25 1.28 Municipals (10yr) 3.35 -0.43 1.58 1.58 1.35 3.65 ¥per$ 147.49 187.16 148.14 High Yield 7A9 -0.28 1.76 1.76 91 16.30 Levels (%) Levels Key Rates 3/7/25 2/28/25 12/31/24 12/31/24 3/7/24 3/7/22 Commod. 3/7/25 12/31/24 3/7/24 2-yr U.S. Treasuries 3.99 3.99 4.25 4.25 4.50 1.55 Oil (WTI) 66.37 72.44 79.81 10-yr U.S. Treasuries 4.32 4.24 4.58 4.58 4.09 1.78 Gasoline 3.08 3.01 (3335) 30-yr U.S. Treasuries 4.62 4.51 478 478 4.25 2.19 Natural Gas 4.31 3.40 1.56 10-yr German Bund 2.83 241 2.35 2.35 2.30 -0.03 Gold 2931 2609 2153 SOFR 4.35 439 449 449 5.31 0.05 _— Silver 32.50 28.11 24.16 3-mo. EURIBOR 2.53 2.46 21 21 3.93 -0.50 Copper 9664 8706 8559 6-mo. CD rate N/A 2.30 2.29 2.29 2

In [27]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

In [28]:
prompt_text = """
  You are responsible for concisely summarizing table or text chunk:

  {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)
summarize_chain = {"element": lambda x: x} | prompt | ChatOpenAI(temperature=0, model="gpt-4") | StrOutputParser()

/tmp/ipykernel_2473644/1321421087.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  summarize_chain = {"element": lambda x: x} | prompt | ChatOpenAI(temperature=0, model="gpt-4") | StrOutputParser()


In [29]:
tables = [i.text for i in table_elements]
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

texts = [i.text for i in text_elements]
text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})

In [30]:
table_summaries

['The text provides a variety of financial data. The NASDAQ is at 18196, with a variety of other statistics provided. Fixed income yields for U.S. Aggregate, U.S. Corporates, Municipals (10yr), and High Yield are given, along with their respective changes. Currency exchange rates for $ per €, $ per £, and ¥ per $ are also provided. Key rates for 2-yr, 10-yr, and 30-yr U.S. Treasuries, 10-yr German Bund, SOFR, 3-mo. EURIBOR, 6-mo. CD rate, 30-yr fixed mortgage, and Prime Rate are listed. Commodity prices for oil (WTI), gasoline, natural gas, gold, silver, copper, and corn are given. The BBG Index is at 255.17.']

In [ ]:
import uuid

from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.schema.document import Document
from langchain.storage import InMemoryStore
from langchain.vectorstores import Chroma

id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings()),
    docstore=InMemoryStore(),
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=s, metadata={id_key: table_ids[i]})
    for i, s in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

# Add images
image_data_list = []
image_summary_list = []
doc_ids = [str(uuid.uuid4()) for _ in image_data_list]
summary_images = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(image_summary_list)
]
retriever.vectorstore.add_documents(summary_images)
retriever.docstore.mset(list(zip(doc_ids, image_data_list)))